In [1]:
import requests
import pandas as pd
import geopandas as gpd
from popframe.preprocessing.level_filler import LevelFiller

DEFAULT_CRS = 4326
URBAN_API = 'http://10.32.1.107:5300'
POPULATION_COUNT_INDICATOR_ID = 1

In [5]:
POPULATION_COUNT_INDICATOR_ID = 1

def get_territories(parent_id : int | None = None, all_levels = False, geometry : bool = False) -> pd.DataFrame | gpd.GeoDataFrame:
    res = requests.get(URBAN_API + f'/api/v1/all_territories{"" if geometry else "_without_geometry"}', {
        'parent_id': parent_id,
        'get_all_levels': all_levels
    })
    res_json = res.json()
    if geometry:
        gdf = gpd.GeoDataFrame.from_features(res_json, crs=DEFAULT_CRS)
        return gdf.set_index('territory_id', drop=True)
    df = pd.DataFrame(res_json)
    return df.set_index('territory_id', drop=True)

def get_territories_population(territories_gdf : gpd.GeoDataFrame, regional_scenario_id : int | None = None):
    res = requests.get(f'{URBAN_API}/api/v1/indicator/{POPULATION_COUNT_INDICATOR_ID}/values')
    res_df = pd.DataFrame(res.json())
    res_df = res_df[res_df['territory_id'].isin(territories_gdf.index)]
    res_df = res_df.groupby('territory_id').agg({'value': 'last'}).rename(columns={'value':'population'})
    return territories_gdf[['geometry']].merge(res_df, how='left', left_index=True, right_index=True)

In [6]:
def _fetch_territories(region_id : int) -> tuple[dict[int, gpd.GeoDataFrame], gpd.GeoDataFrame]:
    # fetch towns
    territories_gdf = get_territories(region_id, all_levels = True, geometry=True)
    territories_gdf['was_point'] = territories_gdf['properties'].apply(lambda p : p['was_point'] if 'was_point' in p else False)
    #filter towns gdf
    towns_gdf = territories_gdf[territories_gdf['was_point']]
    # towns_gdf = await get_territories_population(towns_gdf) # раскоментить если нужно население, если индикатора нет, там будет nan
    #filter units gdf
    units_gdf = territories_gdf[~territories_gdf['was_point']]
    levels = units_gdf['level'].unique()
    # fetch population
    return {level:units_gdf[units_gdf.level == level] for level in levels}, towns_gdf

In [2]:
def get_territories_population(territories_gdf : gpd.GeoDataFrame):
    res = requests.get(f'{URBAN_API}/api/v1/indicator/{POPULATION_COUNT_INDICATOR_ID}/values')
    res_df = pd.DataFrame(res.json())
    res_df = res_df[res_df['territory_id'].isin(territories_gdf.index)]
    res_df = res_df.groupby('territory_id').agg({'value': 'last'}).rename(columns={'value':'population'})
    return territories_gdf[['geometry', 'name']].merge(res_df, left_index=True, right_index=True)

def get_territories(parent_id : int | None = None, all_levels = False, geometry : bool = False) -> pd.DataFrame | gpd.GeoDataFrame:
    res = requests.get(URBAN_API + f'/api/v1/all_territories{"" if geometry else "_without_geometry"}', {
        'parent_id': parent_id,
        'get_all_levels': all_levels
    })
    res_json = res.json()
    if geometry:
        gdf = gpd.GeoDataFrame.from_features(res_json, crs=DEFAULT_CRS)
        return gdf.set_index('territory_id', drop=True)
    df = pd.DataFrame(res_json)
    return df.set_index('territory_id', drop=True)

def load_towns(region_id: int) -> gpd.GeoDataFrame:
    territories_gdf = get_territories(region_id, all_levels = True, geometry=True)
    territories_gdf['was_point'] = territories_gdf['properties'].apply(lambda p : p['was_point'] if 'was_point' in p else False)
    towns_gdf = territories_gdf[territories_gdf['was_point']]
    towns_gdf['geometry'] = towns_gdf['geometry'].representative_point()
    towns_gdf = get_territories_population(towns_gdf) 
    towns_gdf['id'] = towns_gdf.index
    level_filler = LevelFiller(towns=towns_gdf)
    towns = level_filler.fill_levels()
    return towns

In [3]:
region_id = 1
towns = load_towns(region_id)

/Users/mvin/Code/PopFrame_API/.venv/lib/python3.10/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [4]:
towns

,geometry,id,name,population,level
territory_id,,,,,
207,POINT (33.75892 59.36226),207,Болото,10,Малое сельское поселение
208,POINT (33.786 59.47517),208,Большой Остров,68,Малое сельское поселение
209,POINT (33.79236 59.47356),209,Бор,1734,Большое сельское поселение
210,POINT (33.77572 59.44249),210,Бороватое,10,Малое сельское поселение
211,POINT (33.67728 59.32819),211,Бочево,10,Малое сельское поселение
...,...,...,...,...,...
3133,POINT (31.23421 59.17021),3133,Апраксин Бор,313,Среднее сельское поселение
3134,POINT (31.31924 59.18702),3134,Александровка,313,Среднее сельское поселение
3135,POINT (31.47462 59.29408),3135,Большая Горка,313,Среднее сельское поселение


## Применение

In [8]:
units_gdfs, towns_gdf = _fetch_territories(1)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/json/decoder.py:353: RuntimeWarning: coroutine 'get_territories' was never awaited
  obj, end = self.scan_once(s, idx)


In [9]:
units_gdfs.keys()

dict_keys([3, 4])

In [10]:
units_gdfs[3].head()

,geometry,territory_type,parent_id,name,level,properties,admin_center,okato_code,oktmo_code,created_at,updated_at,was_point
territory_id,,,,,,,,,,,,
2,"POLYGON ((34.32834 59.19564, 34.32777 59.19548...","{'territory_type_id': 2, 'name': 'Муниципально...",1,Бокситогорский муниципальный район,3,"{'Малые города': 2, 'Крупные города': 0, 'Числ...",NaN,41203000000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False
10,"POLYGON ((28.98894 59.48069, 28.98604 59.48401...","{'territory_type_id': 2, 'name': 'Муниципально...",1,Волосовский муниципальный район,3,"{'Малые города': 0, 'Крупные города': 0, 'Числ...",NaN,41206000000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False
18,"POLYGON ((32.85314 60.51175, 32.84899 60.50379...","{'territory_type_id': 2, 'name': 'Муниципально...",1,Волховский муниципальный район,3,"{'Малые города': 0, 'Крупные города': 0, 'Числ...",NaN,41209000000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False
34,"MULTIPOLYGON (((30.55161 59.96981, 30.552 59.9...","{'territory_type_id': 2, 'name': 'Муниципально...",1,Всеволожский муниципальный район,3,"{'Малые города': 1, 'Крупные города': 0, 'Числ...",NaN,41212000000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False
54,"POLYGON ((28.9964 60.03638, 28.8588 60.04, 28....","{'territory_type_id': 2, 'name': 'Муниципально...",1,Выборгский муниципальный район,3,"{'Малые города': 0, 'Крупные города': 0, 'Числ...",NaN,41215000000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False


In [11]:
units_gdfs[4].head()

,geometry,territory_type,parent_id,name,level,properties,admin_center,okato_code,oktmo_code,created_at,updated_at,was_point
territory_id,,,,,,,,,,,,
3,"POLYGON ((34.42169 59.68391, 34.42428 59.6715,...","{'territory_type_id': 3, 'name': 'Поселение'}",2,Самойловское сельское поселение,4,"{'Численность населения': 2154, 'Административ...",NaN,41203876000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False
4,"POLYGON ((34.30682 59.71494, 34.30791 59.71295...","{'territory_type_id': 3, 'name': 'Поселение'}",2,Большедворское сельское поселение,4,"{'Численность населения': 1698, 'Административ...",NaN,41203812000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False
5,"POLYGON ((34.14251 59.48759, 34.13616 59.49064...","{'territory_type_id': 3, 'name': 'Поселение'}",2,Пикалевское городское поселение,4,"{'Численность населения': 20169, 'Администрати...",NaN,41440000000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False
6,"POLYGON ((34.05997 59.15087, 34.06007 59.14658...","{'territory_type_id': 3, 'name': 'Поселение'}",2,Борское сельское поселение,4,"{'Численность населения': 3393, 'Административ...",NaN,41203816000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False
7,"POLYGON ((34.06836 59.40315, 34.05535 59.40393...","{'territory_type_id': 3, 'name': 'Поселение'}",2,Бокситогорское городское поселение,4,"{'Численность населения': 15960, 'Администрати...",NaN,41403000000,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,False


In [13]:
towns_gdf.head()

,geometry,territory_type,parent_id,name,level,properties,admin_center,okato_code,oktmo_code,created_at,updated_at,was_point
territory_id,,,,,,,,,,,,
207,"POLYGON ((33.7941 59.36206, 33.79334 59.35856,...","{'territory_type_id': 8, 'name': 'Деревня'}",6,Болото,5,{'was_point': True},NaN,41203816004,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,True
208,"POLYGON ((33.82129 59.47496, 33.82053 59.47146...","{'territory_type_id': 8, 'name': 'Деревня'}",6,Большой Остров,5,{'was_point': True},NaN,41203816020,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,True
209,"POLYGON ((33.82765 59.47334, 33.82689 59.46985...","{'territory_type_id': 8, 'name': 'Деревня'}",6,Бор,5,{'was_point': True},1.0,41203816001,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,True
210,"POLYGON ((33.81098 59.44228, 33.81022 59.43878...","{'territory_type_id': 8, 'name': 'Деревня'}",6,Бороватое,5,{'was_point': True},NaN,41203816005,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,True
211,"POLYGON ((33.71243 59.32801, 33.71168 59.32451...","{'territory_type_id': 8, 'name': 'Деревня'}",6,Бочево,5,{'was_point': True},NaN,41203816003,None,2024-06-16T21:35:40.801621Z,2024-06-16T21:35:40.801621Z,True
